# Stage 1: VGGT → gsplat 3D 재구성 파이프라인

직방/다방 와이드앵글 방 사진 → VGGT로 카메라 포즈/포인트맵 추정 → COLMAP 포맷 export → gsplat으로 3D Gaussian Splatting 학습 → `.ply` 생성

**실행 전 체크리스트**
- 런타임 유형: GPU (A100 or T4 이상, Colab Pro+ 기준 A100 권장)
- 테스트 사진 3~10장 정도를 구글 드라이브의 특정 폴더에 미리 올려두기 (예: `MyDrive/room3d/raw_photos/room_001/`)
- 사진 파일명에 한글/공백 있으면 에러날 수 있으니 `room_001_01.jpg` 식으로 영문+숫자 권장


## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability())


Thu Jul  9 06:23:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
#재시작 필수
!pip install --force-reinstall --no-deps -q "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 132.0 MB/s eta 0:00:00


In [ ]:
# ===== 1. 영상에서 100장 균등 추출 =====
import cv2, os
from pathlib import Path

VIDEO_PATH = "/content/room.mov"
#VIDEO_PATH = "/content/sk304_2.mov"
FRAMES_DIR = "/content/frames"
N_FRAMES = 10

os.makedirs(FRAMES_DIR, exist_ok=True)
for f in Path(FRAMES_DIR).glob("*.jpg"):
    f.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = [int(i * total / N_FRAMES) for i in range(N_FRAMES)]

saved = 0
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(f"{FRAMES_DIR}/frame_{saved:04d}.jpg", frame)
        saved += 1
cap.release()
print(f" {saved}장 추출 (영상 총 {total}프레임 중 균등 샘플링)")

 10장 추출 (영상 총 1611프레임 중 균등 샘플링)


In [1]:
# ===== 1. 내 사진 5장 사용 =====
import os, glob, shutil
from pathlib import Path

SRC_DIR = "/content/my_photos"   # 올려둔 사진 폴더
FRAMES_DIR = "/content/frames"

os.makedirs(FRAMES_DIR, exist_ok=True)
for f in Path(FRAMES_DIR).glob("*.jpg"):
    f.unlink()

exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
src_files = []
for e in exts:
    src_files += glob.glob(os.path.join(SRC_DIR, e))
src_files = sorted(src_files)

saved = 0
for p in src_files:
    dst = f"{FRAMES_DIR}/frame_{saved:04d}.jpg"
    shutil.copy(p, dst)   # png여도 확장자만 jpg로 복사됨 (DUSt3R가 내용으로 읽어 문제없음)
    saved += 1

print(f" {saved}장 준비 (원본 {len(src_files)}장)")

 0장 준비 (원본 0장)


In [2]:
# ===== 2. VGGT 설치 확인 =====
import os, sys

if not os.path.exists("/content/vggt/vggt/models"):
    !rm -rf /content/vggt
    !git clone https://github.com/facebookresearch/vggt.git /content/vggt
    !pip install -q -e /content/vggt

if "/content/vggt" not in sys.path:
    sys.path.insert(0, "/content/vggt")

print("model.py 존재:", os.path.exists("/content/vggt/vggt/models/vggt.py"))

Cloning into '/content/vggt'...
remote: Enumerating objects: 1281, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 1281 (delta 2), reused 1 (delta 1), pack-reused 1273 (from 2)
Receiving objects: 100% (1281/1281), 64.95 MiB | 15.93 MiB/s, done.
Resolving deltas: 100% (589/589), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 42.8 MB/s eta 0:00:00
  Building editable for vggt (pyproject.toml) ... done
model.py 존재: True


In [3]:
# ===== 3. 모델 로드 =====
import torch, glob
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

image_names = sorted(glob.glob(f"{FRAMES_DIR}/*.jpg"))
print(f"입력 이미지: {len(image_names)}장, device: {device}, dtype: {dtype}")

config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.03G [00:00<?, ?B/s]

입력 이미지: 3장, device: cuda, dtype: torch.bfloat16


In [4]:
# ===== 4. 추론 — A100 80GB, 100장은 안전선. 혹시 몰라 폴백만 남겨둠 =====
def run_vggt(names, model, device, dtype):
    images = load_and_preprocess_images(names).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=dtype):
            preds = model(images)
    return preds, images

try:
    predictions, input_images = run_vggt(image_names, model, device, dtype)
    print(f"✅ {len(image_names)}장 추론 성공")
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    fallback_n = N_FRAMES/2
    print(f"⚠️ OOM → {fallback_n}장으로 축소 재시도 (80GB에서 OOM이면 다른 프로세스가 VRAM 점유 중일 가능성 높음)")
    step = len(image_names) / fallback_n
    image_names = [image_names[int(i * step)] for i in range(fallback_n)]
    predictions, input_images = run_vggt(image_names, model, device, dtype)
    print(f"✅ {len(image_names)}장 추론 성공")

/content/vggt/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


✅ 3장 추론 성공


In [5]:
# ===== 5. 포인트 클라우드 추출 + 컬러 매핑 =====
import numpy as np

world_points = predictions["world_points"].float().cpu().numpy()
conf         = predictions["world_points_conf"].float().cpu().numpy()

print("world_points shape:", world_points.shape)
print("conf shape:", conf.shape)

# 배치 차원(맨 앞 1)이 있으면 제거
if world_points.ndim == 5:   # (1, S, H, W, 3)
    world_points = world_points[0]
if conf.ndim == 4:           # (1, S, H, W)
    conf = conf[0]

colors = (input_images.cpu().numpy().transpose(0, 2, 3, 1) * 255)
colors = np.clip(colors, 0, 255).astype(np.uint8)                     # (S,H,W,3)

CONF_THRESH = 1.0
mask = conf > CONF_THRESH

points = world_points[mask]
point_colors = colors[mask]
print(f"필터링 후 포인트 수: {points.shape[0]:,}")

world_points shape: (1, 3, 518, 518, 3)
conf shape: (1, 3, 518, 518)
필터링 후 포인트 수: 804,972


In [6]:
# ===== 5.5. Point Cloud → GLB 저장 (unlit 머티리얼 적용) =====
!pip install -q trimesh pygltflib
import trimesh
from pygltflib import GLTF2
import numpy as np
import os

def to_opengl(points):
    """VGGT 좌표계 -> OpenGL/GLB 좌표계 (Y, Z 반전) — 바닥 정렬 이후 적용"""
    p = points.copy()
    p[:, 1] = -p[:, 1]
    p[:, 2] = -p[:, 2]
    return p

os.makedirs("/content/outputs", exist_ok=True)

# RGBA로 변환 (trimesh PointCloud는 alpha 채널 필요)
colors_rgba = np.hstack([
    point_colors,
    np.full((point_colors.shape[0], 1), 255, dtype=np.uint8)
])

point_cloud = trimesh.points.PointCloud(vertices=to_opengl(points), colors=colors_rgba)
scene = trimesh.Scene([point_cloud])

glb_path = "/content/outputs/vggt_pointcloud_raw.glb"
scene.export(glb_path)

# unlit 처리 (DUSt3R 때와 동일 — Babylon Sandbox 등에서 밝게 뜨는 문제 방지)
gltf = GLTF2().load(glb_path)
for material in gltf.materials:
    if material.extensions is None:
        material.extensions = {}
    material.extensions["KHR_materials_unlit"] = {}
    if material.pbrMetallicRoughness:
        material.pbrMetallicRoughness.metallicFactor = 0.0
        material.pbrMetallicRoughness.roughnessFactor = 1.0
if gltf.extensionsUsed is None:
    gltf.extensionsUsed = []
if "KHR_materials_unlit" not in gltf.extensionsUsed:
    gltf.extensionsUsed.append("KHR_materials_unlit")

unlit_path = "/content/outputs/vggt_pointcloud.glb"
gltf.save(unlit_path)
print(f"{unlit_path} 저장 완료 ({points.shape[0]:,} points)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
/content/outputs/vggt_pointcloud.glb 저장 완료 (804,972 points)


###바닥 정렬 수정해야됨

In [ ]:
# ===== 5.6. 바닥 평면 검출 및 Y-up 정렬 =====
!pip install -q open3d
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(point_colors / 255.0)

# RANSAC으로 가장 큰 평면(바닥일 확률 높음) 검출
plane_model, inliers = pcd.segment_plane(
    distance_threshold=0.02,  # 평면으로 인정할 오차 범위(단위: world_points 스케일)
    ransac_n=3,
    num_iterations=1000
)
a, b, c, d = plane_model
print(f"검출된 평면: {a:.3f}x + {b:.3f}y + {c:.3f}z + {d:.3f} = 0")
print(f"평면 위 포인트 수: {len(inliers):,} / 전체 {len(points):,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 161.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 110.3 MB/s eta 0:00:00
검출된 평면: -0.156x + 0.574y + 0.804z + -1.224 = 0
평면 위 포인트 수: 342,530 / 전체 2,030,560


In [ ]:
# 검출된 평면 기준으로 전체 포인트가 한쪽에 몰려있는지 확인
distances = a * points[:,0] + b * points[:,1] + c * points[:,2] + d
above = np.sum(distances > 0)
below = np.sum(distances < 0)
print(f"평면 위: {above:,} ({above/len(points)*100:.1f}%)  |  평면 아래: {below:,} ({below/len(points)*100:.1f}%)")

평면 위: 189,053 (9.3%)  |  평면 아래: 1,841,503 (90.7%)


In [ ]:
# 평면 법선을 Y축(up)에 맞춰 회전
normal = np.array([a, b, c])
normal = normal / np.linalg.norm(normal)

# 카메라가 보통 바닥을 위에서 내려다보므로 법선이 -Y 방향이면 뒤집기
if normal[1] < 0:
    normal = -normal

target = np.array([0, 1, 0])  # Y-up
rot_axis = np.cross(normal, target)
rot_axis_norm = np.linalg.norm(rot_axis)

if rot_axis_norm > 1e-6:
    rot_axis = rot_axis / rot_axis_norm
    angle = np.arccos(np.clip(np.dot(normal, target), -1, 1))
    R = o3d.geometry.get_rotation_matrix_from_axis_angle(rot_axis * angle)
    pcd.rotate(R, center=(0, 0, 0))

# 바닥을 Y=0으로 이동
rotated_points = np.asarray(pcd.points)
floor_y = np.percentile(rotated_points[:, 1], 1)  # 하위 1% 지점을 바닥으로 간주
pcd.translate((0, -floor_y, 0))

points = np.asarray(pcd.points)
point_colors = (np.asarray(pcd.colors) * 255).astype(np.uint8)
print(" 바닥 정렬 완료 (Y=0 기준)")

 바닥 정렬 완료 (Y=0 기준)


In [ ]:
# ===== 5.5. Point Cloud → GLB 저장 =====
!pip install -q trimesh

import trimesh
import numpy as np
import os

os.makedirs("/content/outputs", exist_ok=True)

# RGBA로 변환 (trimesh PointCloud는 alpha 채널 필요)
colors_rgba = np.hstack([
    point_colors,
    np.full((point_colors.shape[0], 1), 255, dtype=np.uint8)
])

point_cloud = trimesh.points.PointCloud(vertices=points, colors=colors_rgba)
scene = trimesh.Scene([point_cloud])

glb_path = "/content/outputs/vggt_pointcloud_re.glb"
scene.export(glb_path)
print(f" {glb_path} 저장 완료 ({points.shape[0]:,} points)")

 /content/outputs/vggt_pointcloud_re.glb 저장 완료 (2,030,560 points)
